# Pareto Frontier & DFZ Visualization
Reproduces paper Figures and Table 3-5.

Requires all 11 configs to have been trained and evaluated.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
import sys
sys.path.insert(0, '..')

from src.evaluation.pareto import (
    ConfigResult, compute_dfz, find_knee_point,
    ternary_sensitivity_analysis, bootstrap_pareto_stability,
    generate_benchmark_table
)

RESULTS_DIR = Path('../outputs')
S_STAR, T_STAR = 10.0, 300.0

In [ ]:
# Load results (or use paper values for reproduction)
PAPER_RESULTS = [
    dict(config_id='A1', config_name='Basic Aug',   f1=0.506, eod_gender=0.121, eod_ethnicity=0.162, size_mb=27.8, latency_e3_ms=218),
    dict(config_id='A2', config_name='3D Aug M7',   f1=0.946, eod_gender=0.021, eod_ethnicity=0.039, size_mb=27.8, latency_e3_ms=214),
    dict(config_id='A3', config_name='Std.Prune',   f1=0.874, eod_gender=0.083, eod_ethnicity=0.116, size_mb=6.3,  latency_e3_ms=187),
    dict(config_id='B1', config_name='PFP',         f1=0.904, eod_gender=0.021, eod_ethnicity=0.039, size_mb=6.3,  latency_e3_ms=187),
    dict(config_id='B2', config_name='INT8 Quant',  f1=0.919, eod_gender=0.024, eod_ethnicity=0.042, size_mb=7.1,  latency_e3_ms=142),
    dict(config_id='B3', config_name='KD MNv3',     f1=0.891, eod_gender=0.031, eod_ethnicity=0.050, size_mb=4.2,  latency_e3_ms=96),
    dict(config_id='B4', config_name='PFP+INT8',    f1=0.887, eod_gender=0.023, eod_ethnicity=0.040, size_mb=3.1,  latency_e3_ms=91),
    dict(config_id='C1', config_name='M7+StdPrn',   f1=0.921, eod_gender=0.079, eod_ethnicity=0.108, size_mb=6.3,  latency_e3_ms=187),
    dict(config_id='C2', config_name='M7+PFP',      f1=0.934, eod_gender=0.020, eod_ethnicity=0.037, size_mb=6.3,  latency_e3_ms=187),
    dict(config_id='C3', config_name='M7+INT8',     f1=0.938, eod_gender=0.022, eod_ethnicity=0.038, size_mb=7.1,  latency_e3_ms=142),
    dict(config_id='C4', config_name='M1+PFP',      f1=0.871, eod_gender=0.024, eod_ethnicity=0.041, size_mb=6.3,  latency_e3_ms=187),
]

results = []
for d in PAPER_RESULTS:
    l_acc  = 1 - d['f1']
    l_fair = 0.5 * (d['eod_gender'] + d['eod_ethnicity'])
    l_eff  = 0.5*(d['size_mb']/S_STAR) + 0.5*(d['latency_e3_ms']/T_STAR)
    results.append(ConfigResult(
        config_id=d['config_id'], config_name=d['config_name'],
        l_acc=l_acc, l_fair=l_fair, l_eff=l_eff,
        f1=d['f1'], eod_gender=d['eod_gender'], eod_ethnicity=d['eod_ethnicity'],
        size_mb=d['size_mb'], latency_e3_ms=d['latency_e3_ms'],
    ))

print(f'Loaded {len(results)} configurations')

In [ ]:
# Compute Pareto frontier and DFZ
dfz = compute_dfz(results)
knee = find_knee_point(dfz)

print(f'DFZ-qualified: {[c.config_id for c in dfz]}')
print(f'Knee point: {knee.config_id} (||L||_2 = {knee.distance_to_ideal():.3f})')

# Benchmark table (Table 3)
table = generate_benchmark_table(results)
print('\nBenchmark Table:')
display(table)

In [ ]:
# Figure: 3D Pareto frontier (L_acc, L_fair, L_eff)
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

colors = {
    'dfz_knee':  '#2E8B57',   # Green — C2
    'dfz_other': '#5BA85A',   # Light green — other DFZ
    'pareto_only': '#7BB8D4', # Blue — Pareto but not DFZ (A2)
    'dominated':   '#888888', # Gray — non-Pareto
    'excluded':    '#D85A30', # Red — constraint-violating
}

for c in results:
    if c.config_id == 'C2':
        color, size, marker, zorder = colors['dfz_knee'], 200, '*', 10
    elif c.dfz:
        color, size, marker, zorder = colors['dfz_other'], 80, 'o', 5
    elif c.pareto_optimal:
        color, size, marker, zorder = colors['pareto_only'], 80, 's', 4
    elif c.config_id in ('A3', 'C1'):
        color, size, marker, zorder = colors['excluded'], 60, 'x', 3
    else:
        color, size, marker, zorder = colors['dominated'], 40, '^', 2

    ax.scatter(c.l_acc, c.l_fair, c.l_eff,
               c=color, s=size, marker=marker, zorder=zorder)
    ax.text(c.l_acc+0.002, c.l_fair+0.001, c.l_eff+0.02,
            c.config_id, fontsize=8)

ax.set_xlabel('L_acc (1 - F1)', labelpad=10)
ax.set_ylabel('L_fair', labelpad=10)
ax.set_zlabel('L_eff', labelpad=10)
ax.set_title('Pareto Frontier in Normalized Objective Space')

legend_handles = [
    mpatches.Patch(color=colors['dfz_knee'],   label='C2: DFZ knee point'),
    mpatches.Patch(color=colors['dfz_other'],  label='DFZ qualified'),
    mpatches.Patch(color=colors['pareto_only'],label='Pareto-optimal, not DFZ'),
    mpatches.Patch(color=colors['excluded'],   label='EOD constraint violated'),
    mpatches.Patch(color=colors['dominated'],  label='Pareto-dominated'),
]
ax.legend(handles=legend_handles, loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig('pareto_3d.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: pareto_3d.pdf')

In [ ]:
# Figure: EOD comparison bar chart (Table 4 visualization)
fig, ax = plt.subplots(figsize=(10, 6))

config_ids   = [r.config_id for r in results]
eod_g_vals   = [r.eod_gender * 100 for r in results]
eod_e_vals   = [r.eod_ethnicity * 100 for r in results]

x = np.arange(len(config_ids))
w = 0.35

dfz_ids = {c.config_id for c in dfz}
bar_colors_g = ['#D85A30' if v > 10 else ('#2E8B57' if c in dfz_ids else '#5B8FB9')
                for c, v in zip(config_ids, eod_g_vals)]
bar_colors_e = ['#D85A30' if v > 10 else ('#2E8B57' if c in dfz_ids else '#5B8FB9')
                for c, v in zip(config_ids, eod_e_vals)]

ax.barh(x + w/2, eod_g_vals, w, label='EOD_gender', color=bar_colors_g, alpha=0.85)
ax.barh(x - w/2, eod_e_vals, w, label='EOD_ethnicity', color=bar_colors_e, alpha=0.55)
ax.axvline(x=10, color='red', linestyle='--', linewidth=1.5, label='τ_fair = 10%')

ax.set_yticks(x)
ax.set_yticklabels(config_ids)
ax.set_xlabel('EOD (%)')
ax.set_title('EOD_gender and EOD_ethnicity — all 11 configurations')
ax.legend()
ax.invert_yaxis()

# Highlight C2
c2_idx = config_ids.index('C2')
ax.axhspan(c2_idx - 0.5, c2_idx + 0.5, alpha=0.08, color='green')

plt.tight_layout()
plt.savefig('eod_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Ternary sensitivity analysis
sensitivity = ternary_sensitivity_analysis(results)
print(f"n_combinations: {sensitivity['n_combinations']}")
print('Optimal fractions:')
for cid, frac in sorted(sensitivity['optimal_fractions'].items(), key=lambda x: x[1], reverse=True):
    print(f'  {cid}: {frac*100:.1f}%')

# Bootstrap stability
stability = bootstrap_pareto_stability(results, n_bootstrap=1000)
print('\nKnee point stability:')
for cid, frac in sorted(stability['knee_stability'].items(), key=lambda x: x[1], reverse=True):
    print(f'  {cid}: {frac*100:.1f}%')